# DESC 2D continuum with ALCON

DESC hdf5 + NTP.mat -> alcon.dat -> ALCON -> MATLAB


In [ ]:
################## 1. 用户输入区 ##################

# 文件与剖面
EQUILIBRIUM_HDF5 = r'C:\Users\Desktop\equil.hdf5'  # DESC hdf5 平衡；填当前 case。
PROFILE_MAT = r'C:\Users\Desktop\NTP.mat'           # NTP.mat；需含 rhoSample/neSample/niSample/TeSample/TiSample。
ALCON_DIR = r'~/alcon'                                                # ALCON 源码目录。

# 剖面拟合
NE_POLY_DEGREE = 12            # ne(rho) 多项式次数；需小于样本数。
NI_POLY_DEGREE = 12           # ni(rho) 多项式次数；需小于样本数。
TE_POLY_DEGREE = 12            # Te(rho) 多项式次数；需小于样本数。
TI_POLY_DEGREE = 12            # Ti(rho) 多项式次数；需小于样本数。

# 物理模型
ION_MASS_IN_PROTONS = 2.5     # 背景离子质量 mi/mp；H=1，D=2，当前例子用 2.5。
GAMMA_I = 1.75                # 离子压强响应指数；常用 5/3 或 1.75。
GAMMA_E = 1.0                 # 电子压强响应指数；常用 1.0。
FINITE_BETA = 3               # 0: Alfven；1: 慢声近似；2: 声波；3: 完整声波耦合。

# 模数
TOROIDAL_N = 26               # 环向模数 n；当前例子用 30。
M_RANGE_FULL = (11, 100)       # 完整 m 范围；需覆盖 n*q(rho)。
M_HALF_WIDTH = 20             # 每个半径取 m0±该值，m0≈nq；建议 8~15。
NOFFDIAG_MAX = 20             # Fourier 旁带耦合宽度；需 <= ALCON_NFFTCOEF。

# 分辨率
RHO_RANGE = (0.1, 0.9)        # 计算/显示的 rho=sqrt(s) 范围。
ALCON_DATA_RADIAL_POINTS = 2048 # 写 alcon.dat 的径向采样；建议 161~321。
RADIAL_POINTS = 2048           # ALCON 求谱径向点数；建议 401~801。
THETA_POINTS = 1024           # 角向采样点数；建议 >= 2*(ALCON_NFFTCOEF+1)。
ALCON_NFFTCOEF = 64           # 平衡 Fourier 系数最高阶；需 >= NOFFDIAG_MAX。
OMEGA_CUTOFF = 1.1            # 无量纲频率截断；建议 1.0~1.5。


In [ ]:
################## 2. 读取并拟合 NTP ##################

from pathlib import Path
import os, re, sys, tempfile
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

os.environ.setdefault('JAX_PLATFORM_NAME', 'cpu')
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

KEV_TO_ERG = 1.602176634e-9
M_PROTON = 1.67262192595e-27
MU0 = 4e-7 * np.pi


def to_wsl_path(path):
    text = str(path).strip().strip('"').strip("'").replace(chr(92), '/')
    m = re.match(r'^([A-Za-z]):/(.*)$', text)
    return Path(f'/mnt/{m.group(1).lower()}/{m.group(2)}') if m else Path(text).expanduser()


def read_profile_mat(path):
    names = ['rhoSample', 'neSample', 'niSample', 'TeSample', 'TiSample']
    try:
        mat = loadmat(path)
        return tuple(np.ravel(mat[name]).astype(float) for name in names)
    except NotImplementedError:
        import h5py
        with h5py.File(path, 'r') as mat:
            return tuple(np.ravel(np.array(mat[name])).astype(float) for name in names)


def fit_polynomial(x, y, degree, name):
    degree = int(degree)
    if degree < 0:
        raise ValueError(f'{name} polynomial degree must be non-negative.')
    if degree >= len(x):
        raise ValueError(f'{name} polynomial degree must be smaller than sample count.')
    return np.polynomial.polynomial.polyfit(x, y, degree)


def polyval_rho(coeff, rho):
    return np.polynomial.polynomial.polyval(np.asarray(rho), coeff)


profile_path = to_wsl_path(PROFILE_MAT)
if not profile_path.exists():
    raise FileNotFoundError(profile_path)

rho_profile, ne_sample, ni_sample, te_sample, ti_sample = read_profile_mat(profile_path)
order = np.argsort(rho_profile)
rho_profile = rho_profile[order]
ne_sample = ne_sample[order]
ni_sample = ni_sample[order]
te_sample = te_sample[order]
ti_sample = ti_sample[order]

if np.any(~np.isfinite(rho_profile)) or np.any(~np.isfinite(ne_sample)) or np.any(~np.isfinite(ni_sample)) or np.any(~np.isfinite(te_sample)) or np.any(~np.isfinite(ti_sample)):
    raise ValueError('profile mat contains non-finite values.')
if rho_profile[0] < -1e-12 or rho_profile[-1] > 1 + 1e-12:
    print('Warning: rhoSample is expected to be in [0, 1].')

ne_coeff = fit_polynomial(rho_profile, ne_sample, NE_POLY_DEGREE, 'ne')
ni_coeff = fit_polynomial(rho_profile, ni_sample, NI_POLY_DEGREE, 'ni')
te_coeff = fit_polynomial(rho_profile, te_sample, TE_POLY_DEGREE, 'Te')
ti_coeff = fit_polynomial(rho_profile, ti_sample, TI_POLY_DEGREE, 'Ti')
ne_fit = polyval_rho(ne_coeff, rho_profile)
ni_fit = polyval_rho(ni_coeff, rho_profile)
te_fit = polyval_rho(te_coeff, rho_profile)
ti_fit = polyval_rho(ti_coeff, rho_profile)
if ne_coeff[0] <= 0 or ni_coeff[0] <= 0:
    raise ValueError('fitted ne(0) and ni(0) must be positive.')
if np.min(ne_fit) <= 0 or np.min(ni_fit) <= 0 or np.min(te_fit) <= 0 or np.min(ti_fit) <= 0:
    print('Warning: fitted profile has non-positive values.')

PROFILE_FIT = dict(
    mat_file=str(profile_path), rho=rho_profile,
    ne_coeff_rho=ne_coeff, ni_coeff_rho=ni_coeff, te_coeff_rho=te_coeff, ti_coeff_rho=ti_coeff,
    ne0_1e19=float(ne_coeff[0]), ni0_1e19=float(ni_coeff[0]),
)

fig, axes = plt.subplots(2, 2, figsize=(11, 7.0), sharex=True)
items = [
    ('Electron density', r'$n_e\ [10^{19}m^{-3}]$', ne_sample, ne_fit),
    ('Ion density', r'$n_i\ [10^{19}m^{-3}]$', ni_sample, ni_fit),
    ('Electron temperature', r'$T_e\ [keV]$', te_sample, te_fit),
    ('Ion temperature', r'$T_i\ [keV]$', ti_sample, ti_fit),
]
for ax, (title, ylabel, sample, fit) in zip(axes.ravel(), items):
    ax.plot(rho_profile, sample, 'o', ms=3, label='sample')
    ax.plot(rho_profile, fit, '-', lw=1.6, label='fit')
    ax.set_title(title)
    ax.set_xlabel(r'$\rho$')
    ax.set_ylabel(ylabel)
    ax.ticklabel_format(useOffset=False, style='plain')
    ax.grid(alpha=0.2)
axes.ravel()[0].legend(frameon=False)
fig.tight_layout()
plt.show()

print(dict(ne0_1e19=PROFILE_FIT['ne0_1e19'], ni0_1e19=PROFILE_FIT['ni0_1e19'], Ti0_keV=float(polyval_rho(PROFILE_FIT['ti_coeff_rho'], 0.0))))


In [ ]:
################## 3. DESC -> alcon.dat ##################

from desc.io import load
from desc.grid import Grid, LinearGrid

FINITE_BETA = int(FINITE_BETA)
if FINITE_BETA not in {0, 1, 2, 3}:
    raise ValueError('FINITE_BETA must be 0, 1, 2 or 3.')


def load_equilibrium(path):
    obj = load(path)
    return obj if hasattr(obj, 'compute') else obj[-1]


def periodic_resample(angle, values, n):
    angle = np.unwrap(np.asarray(angle))
    values = np.asarray(values)
    order = np.argsort(angle)
    angle, values = angle[order], values[order]
    angle = np.r_[angle, angle[0] + 2*np.pi]
    values = np.r_[values, values[0]]
    target = angle[0] + np.linspace(0.0, 2*np.pi, n, endpoint=False)
    return np.interp(target, angle, values)


def spectral_derivative(values):
    modes = np.fft.fftfreq(values.size, d=1.0 / values.size)
    return np.fft.ifft(1j * modes * np.fft.fft(values)).real


def fft_coeff(values, ncoef):
    return np.fft.rfft(values)[:ncoef + 1] / values.size


if NOFFDIAG_MAX > ALCON_NFFTCOEF:
    raise ValueError('NOFFDIAG_MAX must be <= ALCON_NFFTCOEF.')

eq_path = to_wsl_path(EQUILIBRIUM_HDF5)
if not eq_path.exists():
    raise FileNotFoundError(eq_path)

eq = load_equilibrium(eq_path)
rho_values = np.linspace(float(RHO_RANGE[0]), float(RHO_RANGE[1]), int(ALCON_DATA_RADIAL_POINTS))
theta_values = np.linspace(0.0, 2*np.pi, int(THETA_POINTS), endpoint=False)
profiles = np.zeros((5, rho_values.size))
fftcoefs = np.zeros((int(ALCON_NFFTCOEF) + 1, 5, rho_values.size), dtype=complex)

axis_grid = Grid(np.array([[0.0, 0.0, 0.0]]), sort=False)
axis_data = eq.compute(['R', '|B|'], grid=axis_grid)
axis_R_m = float(np.ravel(axis_data['R'])[0])
axis_B_T = float(np.ravel(axis_data['|B|'])[0])
rr0 = axis_R_m * 100.0
bb0 = axis_B_T * 1.0e4

profile_grid = LinearGrid(rho=rho_values, M=0, N=0, NFP=eq.NFP)
radial = eq.compute(['iota', 'G', 'I'], grid=profile_grid)
iota = np.asarray(radial['iota'], dtype=float).reshape(-1)
q = 1.0 / iota
G_cgs = np.asarray(radial['G'], dtype=float).reshape(-1) * 1.0e6
I_cgs = np.asarray(radial['I'], dtype=float).reshape(-1) * 1.0e6

chunk = 8
for start in range(0, rho_values.size, chunk):
    stop = min(start + chunk, rho_values.size)
    nodes = np.array([[r, t, 0.0] for r in rho_values[start:stop] for t in theta_values])
    data = eq.compute(['theta_PEST', '|B|', '|grad(psi)|^2', 'iota'], grid=Grid(nodes, sort=False))
    nr = stop - start
    theta_pest = np.asarray(data['theta_PEST']).reshape(nr, theta_values.size)
    B_cgs_raw = np.asarray(data['|B|']).reshape(nr, theta_values.size) * 1.0e4
    grad_psi_tor2_raw = np.asarray(data['|grad(psi)|^2']).reshape(nr, theta_values.size)
    iota_raw = np.asarray(data['iota']).reshape(nr, theta_values.size)

    for local, ir in enumerate(range(start, stop)):
        rho = rho_values[ir]
        ne = float(polyval_rho(PROFILE_FIT['ne_coeff_rho'], rho))
        ni = float(polyval_rho(PROFILE_FIT['ni_coeff_rho'], rho))
        te_kev = float(polyval_rho(PROFILE_FIT['te_coeff_rho'], rho))
        ti_kev = float(polyval_rho(PROFILE_FIT['ti_coeff_rho'], rho))
        gq_plus_I = G_cgs[ir] * q[ir] + I_cgs[ir]

        beta_sharp = 4*np.pi * (
            GAMMA_I * ni * 1.0e13 * ti_kev * KEV_TO_ERG
            + GAMMA_E * ne * 1.0e13 * te_kev * KEV_TO_ERG
        ) / bb0**2
        profiles[:, ir] = [rho, q[ir], gq_plus_I, beta_sharp, ni / PROFILE_FIT['ne0_1e19'] * float(ION_MASS_IN_PROTONS)]

        B = periodic_resample(theta_pest[local], B_cgs_raw[local], int(THETA_POINTS))
        grad_psi_pol2 = periodic_resample(
            theta_pest[local], grad_psi_tor2_raw[local] * iota_raw[local]**2 * 1.0e12, int(THETA_POINTS)
        )
        dB_dtheta = spectral_derivative(B)

        H = (grad_psi_pol2 / gq_plus_I) / (bb0 * rr0)
        J = (grad_psi_pol2 * gq_plus_I / B**4) * (bb0 / rr0**3)
        K = (2.0 * G_cgs[ir] * dB_dtheta / B**3) * (bb0 / rr0)
        L = ((beta_sharp + B**2) * gq_plus_I / B**4) * (bb0 / rr0)
        N = (beta_sharp * K**2 / L) / (bb0 * rr0)
        for j, arr in enumerate([H, J, K, L, N]):
            fftcoefs[:, j, ir] = fft_coeff(arr, int(ALCON_NFFTCOEF))

TEMP_DIR = tempfile.TemporaryDirectory(prefix='alcon2D_')
WORK_DIR = Path(TEMP_DIR.name)
alcon_dat = WORK_DIR / 'alcon.dat'
with alcon_dat.open('w', encoding='ascii') as f:
    f.write(f'{rho_values.size} {int(ALCON_NFFTCOEF)} 5 5\n')
    for x in profiles.T.reshape(-1):
        f.write(f'{x:.16e}\n')
    for ir in range(rho_values.size):
        for j in range(5):
            for k in range(int(ALCON_NFFTCOEF) + 1):
                z = fftcoefs[k, j, ir]
                f.write(f'({z.real:.16e},{z.imag:.16e})\n')

f_Ap_kHz = axis_B_T / np.sqrt(MU0 * PROFILE_FIT['ne0_1e19'] * 1.0e19 * M_PROTON) / axis_R_m / (2*np.pi) / 1.0e3
f_A_center_kHz = f_Ap_kHz / np.sqrt(float(ION_MASS_IN_PROTONS) * PROFILE_FIT['ni0_1e19'] / PROFILE_FIT['ne0_1e19'])
INFO = dict(
    alcon_dat=str(alcon_dat), axis_R_m=axis_R_m, axis_B_T=axis_B_T,
    f_Ap_kHz=float(f_Ap_kHz), f_A_center_kHz=float(f_A_center_kHz),
    q_min=float(np.min(q)), q_max=float(np.max(q)),
    beta_min=float(np.min(profiles[3])), beta_max=float(np.max(profiles[3])), finite_beta=int(FINITE_BETA),
    ion_mass_in_protons=float(ION_MASS_IN_PROTONS),
    Te0_keV=float(polyval_rho(PROFILE_FIT['te_coeff_rho'], 0.0)), Ti0_keV=float(polyval_rho(PROFILE_FIT['ti_coeff_rho'], 0.0)),
)
print(INFO)


In [ ]:
################## 4. ALCON 连续谱 ##################

from scipy.linalg import eig

alcon_dir = to_wsl_path(ALCON_DIR)
if str(alcon_dir) not in sys.path:
    sys.path.insert(0, str(alcon_dir))
from alcon_eqdata import EqData
from alcon_solver import Solver


class Input:
    pass


FINITE_BETA = int(FINITE_BETA)
if FINITE_BETA not in {0, 1, 2, 3}:
    raise ValueError('FINITE_BETA must be 0, 1, 2 or 3.')


def make_input(m_range, noffdiag):
    inp = Input()
    inp.eqtype = 'alcon.dat'
    inp.finitebeta = int(FINITE_BETA)
    inp.radrange = [float(RHO_RANGE[0]), float(RHO_RANGE[1])]
    inp.nrad = int(RADIAL_POINTS)
    inp.ntor = int(TOROIDAL_N)
    inp.mpolrange = list(map(int, m_range))
    inp.nmpol = inp.mpolrange[1] - inp.mpolrange[0] + 1
    inp.noffdiag = int(noffdiag)
    inp.ncon = 999
    inp.imreratiocutoff = 0.1
    inp.sigma = -0.5
    inp.omegascale = 1.0
    inp.omegacutoff = float(OMEGA_CUTOFF)
    inp.dirout = str(WORK_DIR)
    inp.erasedirout = True
    inp.n = 1
    inp.v = 0
    return inp


old_cwd = Path.cwd()
os.chdir(WORK_DIR)
base = make_input(M_RANGE_FULL, NOFFDIAG_MAX)
eqd = EqData(base)
rows = []

for irad in range(base.nrad):
    rho = float(eqd.profiles[0, irad])
    q_here = float(eqd.profiles[1, irad])
    m0 = int(round(base.ntor * q_here))
    lo = max(int(M_RANGE_FULL[0]), m0 - int(M_HALF_WIDTH))
    hi = min(int(M_RANGE_FULL[1]), m0 + int(M_HALF_WIDTH))
    if hi - lo + 1 < 5:
        continue

    inp = make_input((lo, hi), min(int(NOFFDIAG_MAX), hi - lo))
    solver = Solver(inp, eqd)
    solver._prepare(irad)
    vals, vecs = eig(solver._matA.toarray(), solver._matB.toarray())
    rho_m = float(eqd.profiles[4, irad])

    for j, val in enumerate(vals):
        omega = np.sqrt(val / rho_m)
        if not np.isfinite(omega.real) or omega.real <= 0:
            continue
        if abs(omega.imag) >= omega.real * inp.imreratiocutoff:
            continue
        omega = float(omega.real * inp.omegascale)
        if inp.omegacutoff > 0 and omega > inp.omegacutoff:
            continue
        dominant = int(np.argmax(np.abs(vecs[:, j])))
        if FINITE_BETA == 3 and dominant >= inp.nmpol:
            continue
        rows.append((rho, omega, lo + dominant, m0))

os.chdir(old_cwd)
rows = np.asarray(rows, dtype=float)
print('points:', len(rows))


In [ ]:
################## 5. 画图 ##################

from scipy.io import savemat
from matplotlib.colors import to_rgb

# 参考频率
REFERENCE_FREQUENCY_KHZ = 86 # 参考频率横线；不用则设 None。
FREQUENCY_RANGE_KHZ = (0.0, 150.0) # 纵坐标范围；None 表示自动。
MIN_FREQUENCY_KHZ = 1.0 # 最小显示频率；None 表示不筛选。
# 推荐 colormap: Reds_r, Greens_r, Blues_r。
CONTINUUM_COLORMAP = 'Greens_r' # 连续谱色图。
REFERENCE_RHO_RANGE = (0.5, 0.6) # 参考频率横线的 rho 范围；None 表示使用 RHO_RANGE。
REFERENCE_LINE_COLOR = 'red' # 参考频率横线颜色。

# MATLAB 导出
MATLAB_EXPORT_BASENAME = 'alcon_continuum' # 导出的 .mat/.m 文件名前缀。
MATLAB_EXPORT_DIR = Path.cwd() # 导出目录；.mat 和 .m 放在同一文件夹即可运行。

fig, ax = plt.subplots(figsize=(9.5, 6.8))
continuum_colors = np.empty((0, 3), dtype=float)
rho_min, rho_max = sorted(map(float, RHO_RANGE))
if FREQUENCY_RANGE_KHZ is not None:
    f_min, f_max = sorted(map(float, FREQUENCY_RANGE_KHZ))
    if f_min < 0.0 or f_max <= f_min:
        raise ValueError('FREQUENCY_RANGE_KHZ must be None or satisfy 0 <= f_min < f_max')
if rows.size:
    frequency_khz_all = rows[:, 1] * f_Ap_kHz
    plot_mask = (rows[:, 0] > rho_min) & (rows[:, 0] < rho_max)
    if MIN_FREQUENCY_KHZ is not None:
        plot_mask &= frequency_khz_all >= float(MIN_FREQUENCY_KHZ)
    if FREQUENCY_RANGE_KHZ is not None:
        plot_mask &= (frequency_khz_all > f_min) & (frequency_khz_all < f_max)
    plot_rows = rows[plot_mask]
    plot_frequency_khz = frequency_khz_all[plot_mask]
else:
    plot_rows = np.empty((0, 4))
    plot_frequency_khz = np.empty(0)
if plot_rows.size:
    m_min, m_max = map(int, M_RANGE_FULL)
    colors = plt.get_cmap(CONTINUUM_COLORMAP)((plot_rows[:, 2] - m_min) / (m_max - m_min))
    continuum_colors = colors[:, :3]
    sc = ax.scatter(plot_rows[:, 0], plot_frequency_khz, s=5.0, c=colors, alpha=0.75, linewidths=0, clip_on=True)
    sc.set_clip_path(ax.patch)
if REFERENCE_FREQUENCY_KHZ is not None:
    if REFERENCE_RHO_RANGE is None:
        reference_rho_range = RHO_RANGE
    else:
        reference_rho_range = REFERENCE_RHO_RANGE
    r_min, r_max = sorted(map(float, reference_rho_range))
    if r_max <= r_min:
        raise ValueError('REFERENCE_RHO_RANGE must be None or satisfy rho_min < rho_max')
    ref_freq = float(REFERENCE_FREQUENCY_KHZ)
    ax.plot((r_min, r_max), (ref_freq, ref_freq), color=REFERENCE_LINE_COLOR, lw=2.0, ls='-', label=f'{REFERENCE_FREQUENCY_KHZ:g} kHz')
ax.set_xlim(rho_min, rho_max)
if FREQUENCY_RANGE_KHZ is not None:
    ax.set_ylim(f_min, f_max)
ax.set_xlabel(r'$r/a$')
ax.set_ylabel(r'$f/\mathrm{kHz}$')
ax.set_title(f'n={TOROIDAL_N} continuum spectrum')
ax.tick_params(direction='in', top=True, right=True)
ax.grid(alpha=0.16)
if REFERENCE_FREQUENCY_KHZ is not None:
    ax.legend(frameon=True, loc='upper right')
fig.tight_layout()
plt.show()

matlab_export_dir = Path(MATLAB_EXPORT_DIR).expanduser()
matlab_export_dir.mkdir(parents=True, exist_ok=True)
matlab_mat_path = matlab_export_dir / f'{MATLAB_EXPORT_BASENAME}.mat'
matlab_script_path = matlab_export_dir / f'{MATLAB_EXPORT_BASENAME}.m'
has_reference_frequency = REFERENCE_FREQUENCY_KHZ is not None
if has_reference_frequency:
    reference_rho_range_export = reference_rho_range
    reference_line_color = to_rgb(REFERENCE_LINE_COLOR)
else:
    reference_rho_range_export = RHO_RANGE
    reference_line_color = (0.0, 0.0, 0.0)
if plot_rows.size:
    rho_plot = plot_rows[:, 0].reshape(-1, 1)
    frequency_khz = plot_frequency_khz.reshape(-1, 1)
else:
    rho_plot = np.empty((0, 1))
    frequency_khz = np.empty((0, 1))
matlab_data = dict(
    rho=rho_plot, frequency_khz=frequency_khz,
    continuum_colors=continuum_colors,
    has_reference_frequency=int(has_reference_frequency),
    reference_frequency_khz=float(REFERENCE_FREQUENCY_KHZ) if has_reference_frequency else np.nan,
    reference_rho_range=np.asarray(reference_rho_range_export, dtype=float),
    reference_line_color=np.asarray(reference_line_color, dtype=float),
    title_text=f'n={TOROIDAL_N} continuum spectrum',
)
savemat(matlab_mat_path, matlab_data)
matlab_script = f'''% ALCON continuum plot exported from continuum2D_ALCON.ipynb
script_dir = fileparts(mfilename('fullpath'));
mat_file = fullfile(script_dir, '{matlab_mat_path.name}');
fig_file = fullfile(script_dir, '{MATLAB_EXPORT_BASENAME}.fig');
data = load(mat_file);

has_reference_frequency = logical(data.has_reference_frequency);
axisSize = 12;
labelSize = 14;
titleSize = 14;
legendSize = 12;
legendLineLength = 18;
lineWidth = 2.0;
markerSize = 5;
markerFaceAlpha = 0.7;
xlimRange = [0.1 0.9];
ylimRange = [0 150];

figure; set(gcf,'position',[1000 500 500 309]);
subplot('position',[0.12 0.15 0.80 0.75])

if ~isempty(data.rho)
    h = scatter(data.rho, data.frequency_khz, markerSize, data.continuum_colors, 'filled', 'MarkerEdgeColor', 'none');
    set(h,'Clipping','on');
    try
        h.MarkerFaceAlpha = markerFaceAlpha;
    catch
    end
end
hold on;
if has_reference_frequency
    refLine = plot(data.reference_rho_range, [data.reference_frequency_khz data.reference_frequency_khz], '-', 'Color', data.reference_line_color, 'LineWidth', lineWidth);
end
xlim(xlimRange);
ylim(ylimRange);

set(gca,'FontName','Times New Roman','FontSize',axisSize,'LineWidth',1);
set(gca,'Layer','top');
set(gca,'xgrid','on','ygrid','on','GridLineWidth',1.0,'GridAlpha',0.3, ...
    'MinorGridAlpha',0.3,'GridColor','k','MinorGridColor','k','GridLineStyle','-','MinorGridLineStyle','-');
xlabel('$r/a$','interpreter','latex','FontSize',labelSize);
ylabel('$f/\\mathrm{{kHz}}$','interpreter','latex','FontSize',labelSize);
title(data.title_text,'interpreter','none','FontSize',titleSize);
set(gca,'XTick',(0.1:0.1:0.9));
set(gca,'YTick',(20:20:140));
set(gca,'ticklength',[0.015 0.015/2]);
ax = gca;
% set(gca,'xminortick','on');
% set(gca,'yminortick','on');
% ax.XAxis.MinorTickValues = [];
% ax.YAxis.MinorTickValues = [];
box on
if has_reference_frequency
    lgd = legend(refLine,sprintf('%g kHz',data.reference_frequency_khz),'Box','on','Location','northeast','FontSize',legendSize);
    try
        lgd.ItemTokenSize = [legendLineLength, lgd.ItemTokenSize(2)];
    catch
    end
end
savefig(gcf, fig_file);
'''
matlab_script_path.write_text(matlab_script, encoding='utf-8')
print('MATLAB mat:', matlab_mat_path)
print('MATLAB script:', matlab_script_path)
